# Анализ данных домена Retail за последние 30 дней

Retail описывает сервис доставки продуктов. Здесь фиксируется полный путь пользователя — от просмотра до оформления заказа, включая добавление в корзину. Контексты включают поиск, каталог, страницы товаров и корзину. https://habr.com/ru/companies/tbank/articles/950696/

In [1]:
import os
import pandas as pd
import numpy as np
import glob
from scipy.stats import zscore

In [2]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/retail/events"

In [3]:
# Получаем список всех файлов .pq
event_files = [f for f in os.listdir(events_dir) if f.endswith('.pq')]

In [4]:
# Отсортируем по номеру дня
event_files_sorted = sorted(event_files, reverse=True)

In [5]:
# Возьмем последние 30 файлов
last_30_files = event_files_sorted[:30]

In [6]:
# Считываем и объединяем данные
dfs = []
for fname in last_30_files:
    filepath = os.path.join(events_dir, fname)
    df = pd.read_parquet(filepath)
    dfs.append(df)

In [7]:
# Объединенный DataFrame за последние 30 дней
recent_events = pd.concat(dfs, ignore_index=True)

In [8]:
users_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/users.pq')
items_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/retail/items.pq')

In [9]:
brands_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/brands.pq', engine='fastparquet')

# Исследуем таблицу recent_events

In [10]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os
0,113011200006078,8923243,fmcg_208146,catalog,view,android
1,113011200032272,12124655,fmcg_1034096,catalog,view,android
2,113011200077336,20658160,fmcg_112697,catalog,view,android
3,113011200112253,85296647,fmcg_618508,catalog,view,android
4,113011200250868,85296647,fmcg_509248,catalog,view,android


In [11]:
recent_events.shape

(47376689, 6)

In [12]:
recent_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47376689 entries, 0 to 47376688
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   timestamp    int64 
 1   user_id      uint64
 2   item_id      object
 3   subdomain    object
 4   action_type  object
 5   os           object
dtypes: int64(1), object(4), uint64(1)
memory usage: 2.1+ GB


In [13]:
print(f'Уникальных пользователей: {recent_events.user_id.nunique()}')
print(f'Уникальных айтемов: {recent_events.item_id.nunique()}')

Уникальных пользователей: 33885
Уникальных айтемов: 185416


#### Распределение операционных систем

In [14]:
recent_events['os'].unique()

array(['android', 'ios'], dtype=object)

Вычислим сколько уникальных юзеров взаимодействовало с операционной системой

In [15]:
recent_events.groupby('os', dropna=False)['user_id'].nunique().sort_values(ascending=False)

os
android    33533
ios        23020
Name: user_id, dtype: int64

Из таблицы видно, что чаще всего используемая ОС - android

In [16]:
recent_events['action_type'].unique()

array(['view', 'click', 'added-to-cart'], dtype=object)

#### Распределение типов действий

Вычислим распределение количества действий в выбранном периоде

In [17]:
recent_events.groupby('action_type', dropna=False)['user_id'].count().sort_values(ascending=False)

action_type
view             46275844
added-to-cart      697232
click              403613
Name: user_id, dtype: int64

Самое частое действие - просмотр. Следующее - добавление в корзину. И менее всего кликов.

Вычислим распределение количества уникальных пользователей по типу событий

In [18]:
recent_events.groupby('action_type', dropna=False)['user_id'].nunique().sort_values(ascending=False)

action_type
added-to-cart    25131
click            24408
view             23457
Name: user_id, dtype: int64

Кажется, что нарушилась воронка. Но здесь могло иметь место такое: уникальные пользователи смотрели товары до того временного промежутка, который мы берем. То есть, их события view просто не попали в наш временной промежуток.

In [19]:
# Пользователи, у которых есть click или added-to-cart, но нет view в нашем периоде
users_with_actions = recent_events[recent_events['action_type'].isin(['click', 'added-to-cart'])]['user_id'].unique()
users_with_views = recent_events[recent_events['action_type'] == 'view']['user_id'].unique()

users_without_views = set(users_with_actions) - set(users_with_views)
print(f"Пользователи с действиями, но без views в анализируемом периоде: {len(users_without_views)}")

Пользователи с действиями, но без views в анализируемом периоде: 10428


#### Распределение событий по контекстам, где они происходили (каталог, поиск, добавление в корзину, страницу товара)

main - страница товара? item - похожий товар?

In [20]:
recent_events['subdomain'].unique()

array(['catalog', 'search', 'item', 'main', 'cart'], dtype=object)

Распределение количества уникальных уникальных пользователей по контекстам

In [21]:
recent_events.groupby('subdomain', dropna=False)['user_id'].nunique().sort_values(ascending=False)

subdomain
catalog    22744
search     21186
main       19022
item       18127
cart       12943
Name: user_id, dtype: int64

Из таблицы видно, что наибольшее количество взаимодействий пользователей с товаром произошло в каталоге.

### Работа со временем. Предположение - время задано в микросекундах со сдвигом

In [22]:
recent_events['timestamp'].min(), recent_events['timestamp'].max()

(np.int64(110505600584983), np.int64(113097599976424))

In [23]:
# конвертируем в datetime (получим примерно 1973 год)
recent_events['datetime_wrong_year'] = pd.to_datetime(recent_events['timestamp'], unit='us')

In [24]:
# желаемая и фактическая даты начала
# .normalize() убирает время, оставляя только дату
desired_start_date = pd.to_datetime('2025-01-01')
actual_start_date = recent_events['datetime_wrong_year'].min().normalize() 

In [25]:
# сдвиг
time_offset = desired_start_date - actual_start_date

In [26]:
# применяем сдвиг ко всем датам
recent_events['datetime_corrected'] = recent_events['datetime_wrong_year'] + time_offset

In [27]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').head())

                timestamp        datetime_wrong_year  \
45982150  110505600584983 1973-07-03 00:00:00.584983   
45982151  110505600711834 1973-07-03 00:00:00.711834   
45982152  110505601068406 1973-07-03 00:00:01.068406   
45982153  110505601223658 1973-07-03 00:00:01.223658   
45982154  110505601354955 1973-07-03 00:00:01.354955   

                 datetime_corrected  
45982150 2025-01-01 00:00:00.584983  
45982151 2025-01-01 00:00:00.711834  
45982152 2025-01-01 00:00:01.068406  
45982153 2025-01-01 00:00:01.223658  
45982154 2025-01-01 00:00:01.354955  


In [28]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').tail())

               timestamp        datetime_wrong_year         datetime_corrected
1238049  113097599527555 1973-08-01 23:59:59.527555 2025-01-30 23:59:59.527555
1238050  113097599558412 1973-08-01 23:59:59.558412 2025-01-30 23:59:59.558412
1238051  113097599723432 1973-08-01 23:59:59.723432 2025-01-30 23:59:59.723432
1238052  113097599841644 1973-08-01 23:59:59.841644 2025-01-30 23:59:59.841644
1238053  113097599976424 1973-08-01 23:59:59.976424 2025-01-30 23:59:59.976424


In [29]:
print(recent_events['datetime_corrected'].max() - recent_events['datetime_corrected'].min())

29 days 23:59:59.391441


Вывод: скорее всего, время действительно задано в микросекундах со сдвигом.

# Исследуем данные из таблицы users_df  (таблица users представлена и согласована во всех доменах)

In [30]:
users_df.head()

,user_id,socdem_cluster,region
0,77309558,21.0,2.0
1,72517894,10.0,90.0
2,86699708,9.0,9.0
3,54241043,17.0,58.0
4,23591057,17.0,4.0


In [31]:
users_df.shape

(3500000, 3)

In [32]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500000 entries, 0 to 3499999
Data columns (total 3 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user_id         uint64 
 1   socdem_cluster  float64
 2   region          float64
dtypes: float64(2), uint64(1)
memory usage: 80.1 MB


In [33]:
users_df['socdem_cluster'].nunique()

21

In [34]:
users_df['region'].nunique()

90

#### Распределние по социо-демографическим кластерам (первые пять наиболее заполненнных)

In [35]:
users_df.groupby('socdem_cluster', dropna=False)['user_id'].nunique().sort_values(ascending=False).head()

socdem_cluster
17.0    473629
20.0    420944
12.0    416287
9.0     340144
21.0    331140
Name: user_id, dtype: int64

#### Распределение по регионам

In [36]:
users_df.groupby('region', dropna=False)['user_id'].nunique().sort_values(ascending=False).head()

region
2.0     551358
60.0    249920
90.0    181623
18.0    137250
24.0    114992
Name: user_id, dtype: int64

# Исследуем данные из таблицы items_df

In [37]:
items_df.head()

,item_id,brand_id,category,subcategory,price,embedding
0,fmcg_10,65693,Office Supplies and Educational Materials,Stationery Paper,-4.406018,"[-0.13557312, 0.049446248, -0.018282967, 0.025..."
1,fmcg_10000,146468,Cleaning Supplies and Everyday Household Items,Cleaning and Detergent Products,-4.205185,"[-0.043275375, -0.035317067, 0.0033219394, -0...."
2,fmcg_1000006,37799,None,None,-4.463660,"[-0.083411045, 0.049153276, -0.08736873, 0.075..."
3,fmcg_1000008,240838,Children's Products and Childcare Items,Food Products,-5.138377,"[-0.055184077, 0.02342301, 0.03789554, 0.11559..."
4,fmcg_1000017,240838,Foodstuffs and Beverages,Sweet Desserts and Confectionery,-3.980383,"[-0.052520722, -0.0063896044, -0.011138491, 0...."


In [38]:
items_df.shape

(250171, 6)

In [39]:
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250171 entries, 0 to 250170
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   item_id      250171 non-null  object 
 1   brand_id     250171 non-null  uint64 
 2   category     240586 non-null  object 
 3   subcategory  240586 non-null  object 
 4   price        223682 non-null  float64
 5   embedding    250171 non-null  object 
dtypes: float64(1), object(4), uint64(1)
memory usage: 11.5+ MB


In [40]:
items_df['category'].unique()

array(['Office Supplies and Educational Materials',
       'Cleaning Supplies and Everyday Household Items', None,
       "Children's Products and Childcare Items",
       'Foodstuffs and Beverages',
       'Automotive Accessories, Spare Parts, and Car Care Products',
       'Cosmetics, Personal Care, and Health Maintenance Products',
       'Pharmaceuticals and Medical Supplies',
       'Home Improvement and Countryside Retreat Essentials', 'Other',
       'Pet Supplies: Food, Accessories, and Grooming Products',
       'Outerwear, Casual Apparel, and Specialized Workwear',
       'Sports Equipment, Gear, and Outdoor Activity Accessories',
       'Hobby, Creative, and Leisure Products',
       'Construction and Renovation Materials, Tools, and Equipment',
       'Fashion Accessories, Tech Add-ons, and Style Enhancements',
       'Household Electrical Appliances',
       'Printed Media: Publications, Textbooks, and Fiction',
       'Electronic Devices and Gadgets', 'Footwear of All Typ

In [41]:
items_df.groupby('category', dropna=False)['item_id'].nunique().sort_values(ascending=False)

category
Foodstuffs and Beverages                                       160128
Cosmetics, Personal Care, and Health Maintenance Products       22998
Home Improvement and Countryside Retreat Essentials             17738
NaN                                                              9585
Children's Products and Childcare Items                          8339
Cleaning Supplies and Everyday Household Items                   7731
Outerwear, Casual Apparel, and Specialized Workwear              4162
Pet Supplies: Food, Accessories, and Grooming Products           3710
Other                                                            3155
Office Supplies and Educational Materials                        2983
Fashion Accessories, Tech Add-ons, and Style Enhancements        1810
Sports Equipment, Gear, and Outdoor Activity Accessories         1289
Hobby, Creative, and Leisure Products                            1252
Automotive Accessories, Spare Parts, and Car Care Products       1215
Pharmaceuti

In [42]:
items_df['subcategory'].nunique()

174

174 уникальные подкатегории

In [43]:
items_df.groupby('subcategory', dropna=False)['item_id'].nunique().sort_values(ascending=False)

subcategory
Dairy Products and Eggs                   22544
Bakery Products                           21650
Sweet Desserts and Confectionery          21644
Meat and Sausage Products                 14334
Beverages and Drinks                      10893
                                          ...  
Remote-Controlled Toys and 3D Modeling        1
Built-In Household Appliances                 1
Esoteric Practices and Divination             1
Wrist, Wall, and Other Watches                1
Business Accessories                          1
Name: item_id, Length: 175, dtype: int64

# Посмотрим на данные из таблицы brands_df (таблица brands представлена и согласована во всех доменах)

In [44]:
brands_df.head()

,brand_id,embedding
0,4,None
1,34,None
2,45,None
3,46,None
4,51,None


In [45]:
brands_df.shape

(24513, 2)

In [46]:
brands_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24513 entries, 0 to 24512
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   brand_id   24513 non-null  uint64
 1   embedding  6559 non-null   object
dtypes: object(1), uint64(1)
memory usage: 383.1+ KB


# Анализ пропусков 

### Пропущенные значения в датасете users_df (совпадает во всех доменах)

In [47]:
users_df.isna().sum()

user_id               0
socdem_cluster     5153
region            58917
dtype: int64

In [48]:
def missing_value_rate(df, column):
  print(f'Доля пропущенных значений в колонке {column}:  {(df[column].isna().sum() / users_df.shape[0]):.4f}')

In [49]:
for column in users_df.columns:
  missing_value_rate(users_df, column)

Доля пропущенных значений в колонке user_id:  0.0000
Доля пропущенных значений в колонке socdem_cluster:  0.0015
Доля пропущенных значений в колонке region:  0.0168


Возможные причины возникновения пропусков:

* Необязательные поля при регистрации

* Система не смогла определить местоположение пользователя, возможно пользователь не дал разрешение на определение местоположения

* Технические ошибки при сборе данных

In [50]:
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0])
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0] / users_df.shape[0])

4667
0.0013334285714285713


Итого у нас есть 4667 записей пользователей, где пропуски в обеих колонках socdem_cluster и region. Это около 1,3%.

In [51]:
# проверим дубликаты
users_df.duplicated().sum()

np.int64(0)

In [52]:
print(f'Минимальное значение в колонках socdem_cluster и region соответвенно: {users_df['socdem_cluster'].min()} и {users_df['region'].min()}')

Минимальное значение в колонках socdem_cluster и region соответвенно: 0.0 и 0.0


### Возможная стратегия работы с пропусками

Процент пропусков очень мал. 

* Технически можно удалить строки с пропущенными значениями в таблице users_df. Но тогда мы должны будем удалить этих пользователей из таблицы events_df, которую мы будем использовать для посторения матрицы взаимодействий и уже на этой основе строить прогноз. 

* Колонки socdem_cluster и region это категориальные данные, закодированные числами. Можно заполнить пропущенные значения специальным зарезервированным числом, которое не встречается среди реальных кодов регионов и кластеров. Таким образом, мы сохраним всю историю взаимодействий c этими айтемами в events_df. Можно использовать число -1, так как минимальное значение в указанных колонках равно ноль.

## Пропущенные значения в items_df

In [53]:
items_df.isna().sum()

item_id            0
brand_id           0
category        9585
subcategory     9585
price          26489
embedding          0
dtype: int64

In [54]:
for column in items_df.columns:
  missing_value_rate(items_df, column)

Доля пропущенных значений в колонке item_id:  0.0000
Доля пропущенных значений в колонке brand_id:  0.0000
Доля пропущенных значений в колонке category:  0.0027
Доля пропущенных значений в колонке subcategory:  0.0027
Доля пропущенных значений в колонке price:  0.0076
Доля пропущенных значений в колонке embedding:  0.0000


### Возможные причины возникновения пропусков

* Для `category` и `subcategory` - товар могли добавить в базу данных, но еще не успели присвоить ему категорию.

* Некоторые товары могут не подходить ни под одну из существующих категорий, и поле остается пустым.

* Поставшики прислали неполную информацию.

* Для `price` - это могут быть товары не для продажи, а, например, промо-материалы.

* Некоторые товары могут не иметь публичной цены, а она узнается по запросу.

* Часть комплекта: товар может продаваться только в составе набора, и у него нет индивидуальной цены.

### Возможная стратегия работы с пропусками

* в колонках `category` и `subcategory` (категориальные данные) заменить на 'unknown'.Таким образом, мы сохраняем всю его ценную историю взаимодействий в `events_df`.
  
* Это лучше, чем заполнять самой частой категорией, так как это может внести смещение в данные.

* Колонка `price`. Можно заполнить нулем, если мы можем предположить, что товар, например, акционный, бесплатный. Но это частный случай и у нас нет данных для выдвижения такого предположения. Лучше заполнить медианой. Медиана более устойчива к выбросам, поэтому она лучше отражает "типичную" цену.

## Пропущенные значения в events_df

In [55]:
recent_events.isna().sum()

timestamp              0
user_id                0
item_id                0
subdomain              0
action_type            0
os                     0
datetime_wrong_year    0
datetime_corrected     0
dtype: int64

In [56]:
recent_events['subdomain'].unique()

array(['catalog', 'search', 'item', 'main', 'cart'], dtype=object)

Пропущенных значений в events нет.

## Пропущенные значения в brands_df (совпадает во всех доменах)

In [57]:
brands_df.isna().sum()

brand_id         0
embedding    17954
dtype: int64

In [58]:
print(f'Доля пропущенных значений в колонке embedding:  {(brands_df['embedding'].isna().sum() / brands_df.shape[0]):.4f}')

Доля пропущенных значений в колонке embedding:  0.7324


Более 73% брендов не имеют эмбеддинга. Возможные причины:

* Пропущенные эмбеддинги могут говорить о холодных брендах, а могут и не говорить.

* Эмбеддинги могут пересчитывать не в реальном времени, а, например, раз в неделю. Пропуски могут соответствовать брендам, которые были добавлены после последнего обновления и еще ждут своей очереди на обработку.

### Возможная стратегия работы с пропусками

* Можно удалить колонку

* Можно оставить и попробовать что-то сделать в рамках нейросетевой модели

* Можно посмотреть временную связь на всех данных

**Заметка про работу с холодным стартом на будущее** (из лекция https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla)

* Рекомендовать популярные

* Построить модель на соцдеме или других данных (Например, женщинам рекомендовать женскую одежду, мужчинам мужскую и тд). То есть построить набор популярных товаров, характерных для этого соцдема.


* Провести онбординг (некоторые сервисы при регистрации спрашивают, укажите что вам нравится)

# Анализ до заполнения пропусков

### Анализ users_id (совпадает во всех доменах)

In [59]:
print(f'Уникальных пользователей: {users_df['user_id'].nunique()}')
print(f'Уникальных регионов: {users_df['region'].nunique()}')
print(f'Уникальных социо-демографических кластеров: {users_df['socdem_cluster'].nunique()}')

Уникальных пользователей: 3500000
Уникальных регионов: 90
Уникальных социо-демографических кластеров: 21


In [60]:
# Посмотрим, сколько пользователей в каждом регионе
users_df['region'].value_counts(dropna=False)

region
2.0     551358
60.0    249920
90.0    181623
18.0    137250
24.0    114992
         ...  
65.0      1687
30.0      1125
69.0       915
39.0       836
20.0       338
Name: count, Length: 91, dtype: int64

In [61]:
# Посмотрим, сколько пользователей в каждом кластере
users_df['socdem_cluster'].value_counts(dropna=False)

socdem_cluster
17.0    473629
20.0    420944
12.0    416287
9.0     340144
21.0    331140
10.0    266045
19.0    261186
4.0     254800
0.0     205299
5.0     148037
7.0     140750
18.0    105353
13.0     40572
6.0      36130
1.0      18629
2.0      12134
16.0      9859
8.0       5825
NaN       5153
3.0       3900
11.0      2637
15.0      1547
Name: count, dtype: int64

In [62]:
# Преобразуем столбцы в категориальный тип
users_df_with_cat = users_df.copy()
users_df_with_cat['socdem_cluster'] = users_df['socdem_cluster'].astype('category')
users_df_with_cat['region'] = users_df['region'].astype('category')

In [63]:
# И применим describe
print(users_df_with_cat[['socdem_cluster', 'region']].describe())

        socdem_cluster     region
count        3494847.0  3441083.0
unique            21.0       90.0
top               17.0        2.0
freq          473629.0   551358.0


### Анализ items_df

In [64]:
dist_without_na = items_df['price'].describe()
print(dist_without_na.apply(lambda x: f'{x:.5f}'))

count    223682.00000
mean         -3.76859
std           1.32461
min         -10.00000
25%          -4.79381
50%          -3.87512
75%          -2.92425
max           7.74009
Name: price, dtype: object


* Из таблицы видно, что среднее значение и медиана очень близки друг к другу, распределение практически симметричное.
  
* Центр распределения смещен в минус

* Основная масса данных лежит в диапазоне между -4.79 и -2.92 

Так как Retail это поведение в сервисе доставки продуктов: от просмотра до оформления заказа. То возможно, что сильно дорогие товары это срочные/экспресс доставки (если в стоимость айтема закладывается тариф доставки).

Применим метод z-score для поиска выбросов

In [65]:
price_without_na = items_df['price'].dropna()

In [66]:
z_scores = zscore(price_without_na) 

In [67]:
z_scores

array([-0.48121831, -0.32960082, -0.52473405, ...,  1.34225573,
        1.03949089,  0.39600335], shape=(223682,))

In [68]:
# Находим индексы выбросов в данных без пропусков.
outlier_indices = np.where(np.abs(z_scores) > 3)[0]

In [69]:
print(f"Найдено выбросов: {len(outlier_indices)}")

Найдено выбросов: 1238


In [70]:
# Смотрим на сами значения выбросов
outliers = price_without_na.iloc[outlier_indices]
print("\nПримеры найденных выбросов:")
print(outliers.sort_values())


Примеры найденных выбросов:
11963    -10.000000
37886      0.206004
40673      0.206231
227438     0.207101
99536      0.208806
            ...    
93556      5.407452
182077     5.424673
57228      5.503918
50637      7.676046
68501      7.740092
Name: price, Length: 1238, dtype: float64


Так как Retail это поведение в сервисе доставки продуктов: от просмотра до оформления заказа. То возможно, что сильно дорогие товары это срочные/экспресс доставки (если в стоимость айтема закладывается тариф доставки).

# Покажем варианты заполнения пропусков

## Обработка users_df (совпадает во всех доменах)
В таблице users_df заполняем пропуски в закодированных категориальных признаках region и socdem_cluster числом -1. Мы это можем сделать, так как минимальные значения этих признаков равны нулю.

In [71]:
users_df['region'] = users_df['region'].fillna(-1)
users_df['socdem_cluster'] = users_df['socdem_cluster'].fillna(-1)

In [72]:
# Меняем тип данных на целочисленный
users_df['region'] = users_df['region'].astype(int)
users_df['socdem_cluster'] = users_df['socdem_cluster'].astype(int)

In [73]:
print('Пропуски в users_df после обработки:')
print(users_df.isnull().sum())

Пропуски в users_df после обработки:
user_id           0
socdem_cluster    0
region            0
dtype: int64


## Обработка items_df

In [74]:
# Заполняем категориальные пропуски строкой 'unknown'
items_df['category'] = items_df['category'].fillna('unknown')
items_df['subcategory'] = items_df['subcategory'].fillna('unknown')

In [75]:
# Рассчитываем медианную цену
median_price = items_df['price'].median()
print(f'Медианная цена для заполнения пропусков: {median_price}')

Медианная цена для заполнения пропусков: -3.875121896478133


In [76]:
items_df['price'].min()

np.float64(-10.0)

В признаке price присутствуют отрицательные значения. Это не выбросы. В описании дата сета price - представляет собой денежную стоимость взаимодействия. Цены в каталогах переведены в промежуток от −10 до 10. 

In [77]:
# Заполняем пропуски в цене медианой
items_df['price'] = items_df['price'].fillna(median_price)

In [78]:
print("Пропуски в items_df после обработки:")
print(items_df.isnull().sum())

Пропуски в items_df после обработки:
item_id        0
brand_id       0
category       0
subcategory    0
price          0
embedding      0
dtype: int64


## Обработка brands_df

Пропуски в embedding не трогаем.

## Анализ цен  из items_df после заполнения пропусков

In [79]:
desc = items_df['price'].describe()
print(desc.apply(lambda x: f'{x:.5f}'))

count    250171.00000
mean         -3.77987
std           1.25295
min         -10.00000
25%          -4.69678
50%          -3.87512
75%          -3.05238
max           7.74009
Name: price, dtype: object


* Из таблицы видно, что среднее значение и медиана очень близки друг к другу, распределение практически симметричное.

* Цены сосредоточены в диапазоне -4.7 до -3.05

In [80]:
# Сравним с распределением цен до заполнения пропусков
print(dist_without_na.apply(lambda x: f'{x:.5f}'))

count    223682.00000
mean         -3.76859
std           1.32461
min         -10.00000
25%          -4.79381
50%          -3.87512
75%          -2.92425
max           7.74009
Name: price, dtype: object


Заполнение пропусков медианой прошло успешно и оказало минимальное, предсказуемое влияние на общую картину распределения цен 

### Анализ events_df

In [81]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os,datetime_wrong_year,datetime_corrected
0,113011200006078,8923243,fmcg_208146,catalog,view,android,1973-08-01 00:00:00.006078,2025-01-30 00:00:00.006078
1,113011200032272,12124655,fmcg_1034096,catalog,view,android,1973-08-01 00:00:00.032272,2025-01-30 00:00:00.032272
2,113011200077336,20658160,fmcg_112697,catalog,view,android,1973-08-01 00:00:00.077336,2025-01-30 00:00:00.077336
3,113011200112253,85296647,fmcg_618508,catalog,view,android,1973-08-01 00:00:00.112253,2025-01-30 00:00:00.112253
4,113011200250868,85296647,fmcg_509248,catalog,view,android,1973-08-01 00:00:00.250868,2025-01-30 00:00:00.250868


In [82]:
recent_events['action_type'].unique()

array(['view', 'click', 'added-to-cart'], dtype=object)

#### Определение самых популярных товаров

События типа 'view' фиксируют только факт показа товара пользователю, и многие из них могут быть случайными или малозначимыми (пользователь просто пролистал страницу, товар попал в рекомендацию, но не заинтересовал). Поэтому топ по просмотрам покажет товары, которые чаще всего показывались, но не обязательно были востребованы.

**Важное из лекции** https://vkvideo.ru/video-123851409_456239385?list=ln-fi0AbfTmgwRkUuI0Ai :

* Популярность не считается по просмотрам. Потому что просмотеры зависят от самой рекомендательной системы, и не зависят от дейтсвий пользователя.

Для выявления действительно популярных и интересных товаров лучше использовать более сильные сигналы, такие как:

'click' — пользователь проявил явный интерес, кликнув на товар.

'like' — проявил позитивную реакцию, добавил в избранное.

'clickout' — намерение к покупке, переход к оформлению или внешнему сайту.

'added-to-cart' - добавление в корзину

In [83]:
# Вспомним, какие есть типы взаимодействий
recent_events['action_type'].unique()

array(['view', 'click', 'added-to-cart'], dtype=object)

In [84]:
# Берем только клики и считаем количества кликов для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'click']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по кликам
print(top_popular_items.head(10))

item_id
fmcg_514296    495
fmcg_651828    431
fmcg_346278    349
fmcg_160165    346
fmcg_450490    313
fmcg_671193    305
fmcg_722287    296
fmcg_292617    290
fmcg_335947    252
fmcg_250877    242
Name: count, dtype: int64


In [85]:
# Берем только добавление в корзину
top_popular_items = recent_events[recent_events['action_type'] == 'added-to-cart']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по лайкам
print(top_popular_items.head(10))

item_id
fmcg_651828    2417
fmcg_514296    2246
fmcg_450490    1577
fmcg_292617    1485
fmcg_722287    1360
fmcg_250877    1328
fmcg_335947    1280
fmcg_671193    1259
fmcg_160165    1125
fmcg_510033    1066
Name: count, dtype: int64


Создадим столбец label. Действию view в колонке label поставим в соответсвие 0, а действиям click и add-to-card поставим в соответсвие 1.

In [86]:
recent_events['label'] = recent_events['action_type'].map({'view':0, 'click':1, 'added-to-cart':1})

In [87]:
recent_events.groupby('action_type')['label'].mean()

action_type
added-to-cart    1.0
click            1.0
view             0.0
Name: label, dtype: float64

In [88]:
# Популярность по количеству уникальных пользователей. 
# Именно такой способ расчета популярности часто используется в рекомендательных системах

# Считаем количество уникальных пользователей, совершивших значимые действия для каждого товара
label_1_recent_events = recent_events[recent_events['label'] == 1]
unique_user_popularity = label_1_recent_events.groupby('item_id')['user_id'].nunique().sort_values(ascending=False)

print(unique_user_popularity.head(10))

item_id
fmcg_651828    1988
fmcg_514296    1854
fmcg_450490    1379
fmcg_292617    1328
fmcg_250877    1223
fmcg_722287    1203
fmcg_671193    1131
fmcg_335947    1127
fmcg_160165    1049
fmcg_510033     943
Name: user_id, dtype: int64


# Взаимосвязи количественной и категориальной переменных

В датасете одна основная количественная переменная - `price`

In [89]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os,datetime_wrong_year,datetime_corrected,label
0,113011200006078,8923243,fmcg_208146,catalog,view,android,1973-08-01 00:00:00.006078,2025-01-30 00:00:00.006078,0
1,113011200032272,12124655,fmcg_1034096,catalog,view,android,1973-08-01 00:00:00.032272,2025-01-30 00:00:00.032272,0
2,113011200077336,20658160,fmcg_112697,catalog,view,android,1973-08-01 00:00:00.077336,2025-01-30 00:00:00.077336,0
3,113011200112253,85296647,fmcg_618508,catalog,view,android,1973-08-01 00:00:00.112253,2025-01-30 00:00:00.112253,0
4,113011200250868,85296647,fmcg_509248,catalog,view,android,1973-08-01 00:00:00.250868,2025-01-30 00:00:00.250868,0


In [90]:
users_df.head()

,user_id,socdem_cluster,region
0,77309558,21,2
1,72517894,10,90
2,86699708,9,9
3,54241043,17,58
4,23591057,17,4


In [91]:
items_df.head()

,item_id,brand_id,category,subcategory,price,embedding
0,fmcg_10,65693,Office Supplies and Educational Materials,Stationery Paper,-4.406018,"[-0.13557312, 0.049446248, -0.018282967, 0.025..."
1,fmcg_10000,146468,Cleaning Supplies and Everyday Household Items,Cleaning and Detergent Products,-4.205185,"[-0.043275375, -0.035317067, 0.0033219394, -0...."
2,fmcg_1000006,37799,unknown,unknown,-4.463660,"[-0.083411045, 0.049153276, -0.08736873, 0.075..."
3,fmcg_1000008,240838,Children's Products and Childcare Items,Food Products,-5.138377,"[-0.055184077, 0.02342301, 0.03789554, 0.11559..."
4,fmcg_1000017,240838,Foodstuffs and Beverages,Sweet Desserts and Confectionery,-3.980383,"[-0.052520722, -0.0063896044, -0.011138491, 0...."


In [92]:
# Рассчитываем среднюю, медианную цену и др. для каждого бренда
price_by_brand = items_df.groupby('brand_id')['price'].describe()
print(price_by_brand.sort_values(by='mean', ascending=False))

            count      mean       std        min       25%       50%  \
brand_id                                                               
25989     14965.0 -3.341128  1.078850 -10.000000 -3.876208 -3.405161   
60434     17661.0 -3.428873  1.000240  -7.528375 -3.947548 -3.516903   
240838    68219.0 -3.500516  1.402137  -6.908911 -4.412595 -3.855937   
65693     42875.0 -3.647454  1.194115  -7.469679 -4.447224 -3.828722   
185133        3.0 -3.875122  0.000000  -3.875122 -3.875122 -3.875122   
37799     57656.0 -4.120575  1.184131  -7.499768 -5.001534 -4.058834   
146468    45176.0 -4.142104  1.080616  -7.453211 -4.964498 -4.093348   
48632      3616.0 -4.192399  1.108305  -6.805468 -4.980283 -4.021621   

               75%       max  
brand_id                      
25989    -2.629785  0.951477  
60434    -2.813930  1.909567  
240838   -2.707129  5.503918  
65693    -2.909179  7.740092  
185133   -3.875122 -3.875122  
37799    -3.442269  3.551214  
146468   -3.459760  1.605034  


In [93]:
# Рассчитываем среднюю, медианную цену и др. для каждой категории
price_by_brand = items_df.groupby('category')['price'].describe()
print(price_by_brand.sort_values(by='mean', ascending=False).head())

                                                     count      mean  \
category                                                               
Home/Office Furniture and Interior Decor             119.0 -1.118252   
Household Electrical Appliances                      822.0 -1.467104   
Footwear of All Types                                219.0 -2.041007   
Sports Equipment, Gear, and Outdoor Activity Ac...  1289.0 -2.515037   
Automotive Accessories, Spare Parts, and Car Ca...  1215.0 -2.810913   

                                                         std       min  \
category                                                                 
Home/Office Furniture and Interior Decor            2.068509 -6.192168   
Household Electrical Appliances                     1.913165 -6.250398   
Footwear of All Types                               1.169580 -4.903785   
Sports Equipment, Gear, and Outdoor Activity Ac...  1.932254 -5.987592   
Automotive Accessories, Spare Parts, and Car Ca... 

In [94]:
# Объединяем таблицу событий с данными о пользователях
merged_df = pd.merge(recent_events, users_df, on='user_id', how='left')

# присоединяем данные о товарах
merged_df = pd.merge(merged_df, items_df, on='item_id', how='left')

In [95]:
# Рассчитываем описательные статистики цен для каждого соц-дем кластера
price_by_cluster = merged_df.groupby('socdem_cluster', observed=True)['price'].describe()

# Выводим кластеры, предпочитающие товары с высокой средней ценой
print("Статистики цен по социально-демографическим кластерам:")
print(price_by_cluster.sort_values(by='mean', ascending=False))

Статистики цен по социально-демографическим кластерам:
                    count      mean       std        min       25%       50%  \
socdem_cluster                                                                 
 6                   22.0 -3.115970  1.713397  -5.727504 -4.867572 -2.902372   
 13                  10.0 -3.182895  2.005357  -5.950385 -4.036265 -3.446598   
 15                3815.0 -3.759192  0.958470  -6.468070 -4.368589 -3.828124   
 7              1210685.0 -3.795282  1.193002  -7.459058 -4.680559 -3.875122   
 11               13172.0 -3.828578  1.160154  -6.416693 -4.720674 -3.875122   
 17             6248141.0 -3.844422  1.172526  -7.459058 -4.690335 -3.887305   
 10             5545440.0 -3.875076  1.199479  -7.459058 -4.744596 -3.921337   
 8                18894.0 -3.876476  0.938880  -6.732774 -4.595824 -3.875122   
 0              3832581.0 -3.879136  1.221005  -7.459058 -4.755425 -3.944962   
 9              7553373.0 -3.891246  1.156936 -10.000000 -4.74025

# Целевая переменная

На основе семинара-лекции: https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla


* У нас есть список релевантных пользователю товаров в обучающей выборке (то есть  айдишники товаров, с которым пользователь сделал значимые интеракции на обучающем промежутке времени). И мы хотим предсказать айдишники товаров, с которыми пользователь провзаимодейтсвует на тестовом (отложенном) периоде времени. Для подсчета метрик у нас должны быть ground truth айтемы, про которые мы знаем, что они релевантны пользователю и мы их будем сравнивать с предсказанными. Ground truth айтемы в рекомендательных системах из данного датасета обычно выбираются как реальные положительные взаимодействия пользователя с айтемами в тестовом временном окне.


* Целевая переменная (таргет) — это список или множество айтемов, с которыми пользователь реально взаимодействовал в тестовом (отложенном) временном промежутке.

* Таргет формируется из положительных (значимых) взаимодействий пользователя с айтемами на отложенном периоде, например, кликов, переходов к покупке, лайков и т.п.

* Эти реальные положительные взаимодействия и есть ground truth айтемы, с которыми сравниваются предсказания модели.

* В обучающей выборке у нас есть история взаимодействий пользователя с релевантными товарами за период до теста - на её основе строится модель.

* В тесте модель должна предсказать айтемы, которые user выберет/кликнет/купит. Это и есть задача.

Иными словами, таргет это бинарная метка релевантности айтема пользователю в тестовом периоде, выраженная в виде множества ground truth айтемов. 